In [1]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model = "openai/gpt-oss-20b")

In [4]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explaination : str

In [5]:
def generate_joke(state : JokeState):
    topic = state['topic']
    
    prompt = f"Give me a funny joke on the topic : {topic}."
    joke = model.invoke(prompt).content
    
    return {'joke':joke}

def explain_joke(state : JokeState):
    joke = state['joke']
    
    prompt = f"Explain this joke to me in a simple way : {joke}."
    explaination = model.invoke(prompt).content
    
    return {'explaination':explaination}

In [6]:
graph = StateGraph(JokeState)

graph.add_node("joke_generator",generate_joke)
graph.add_node("joke_explainer",explain_joke)

graph.add_edge(START,"joke_generator")
graph.add_edge("joke_generator","joke_explainer")
graph.add_edge('joke_explainer',END)

In [8]:
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer = checkpointer)

config1 = {'configurable':{'thread_id':'1'}}

workflow.invoke({'topic':'cold drink'},config = config1)

{'topic': 'cold drink',
 'joke': 'Why did the soda apply for a job?  \nBecause it heard the office had great *cool* vibes and wanted to *pop* into the team!',
 'explaination': '**What’s going on in the joke?**\n\n1. **Soda = a “pop”**  \n   – In many places a fizzy drink is called a *pop* (e.g., “Coca‑Cola pop”).  \n   – So when the joke says the soda wants to “pop into the team,” it’s literally using the word “pop” for both the drink and the action of joining.\n\n2. **“Cool vibes”**  \n   – “Cool” can mean both *low temperature* (which soda is usually chilled) and *awesome, relaxed atmosphere*.  \n   – The office is described as having “great cool vibes,” which plays on the idea that a soda would like a cold, chill environment.\n\n3. **Absurd image**  \n   – The idea of a soda (a non‑living drink) going to work and applying for a job is silly and unexpected. That surprise adds to the humor.\n\n**So the joke is a pun:**\n\n- The soda is attracted to the office because it’s *cool* (cold

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'cold drink', 'joke': 'Why did the soda apply for a job?  \nBecause it heard the office had great *cool* vibes and wanted to *pop* into the team!', 'explaination': '**What’s going on in the joke?**\n\n1. **Soda = a “pop”**  \n   – In many places a fizzy drink is called a *pop* (e.g., “Coca‑Cola pop”).  \n   – So when the joke says the soda wants to “pop into the team,” it’s literally using the word “pop” for both the drink and the action of joining.\n\n2. **“Cool vibes”**  \n   – “Cool” can mean both *low temperature* (which soda is usually chilled) and *awesome, relaxed atmosphere*.  \n   – The office is described as having “great cool vibes,” which plays on the idea that a soda would like a cold, chill environment.\n\n3. **Absurd image**  \n   – The idea of a soda (a non‑living drink) going to work and applying for a job is silly and unexpected. That surprise adds to the humor.\n\n**So the joke is a pun:**\n\n- The soda is attracted to the office becaus

In [11]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'cold drink', 'joke': 'Why did the soda apply for a job?  \nBecause it heard the office had great *cool* vibes and wanted to *pop* into the team!', 'explaination': '**What’s going on in the joke?**\n\n1. **Soda = a “pop”**  \n   – In many places a fizzy drink is called a *pop* (e.g., “Coca‑Cola pop”).  \n   – So when the joke says the soda wants to “pop into the team,” it’s literally using the word “pop” for both the drink and the action of joining.\n\n2. **“Cool vibes”**  \n   – “Cool” can mean both *low temperature* (which soda is usually chilled) and *awesome, relaxed atmosphere*.  \n   – The office is described as having “great cool vibes,” which plays on the idea that a soda would like a cold, chill environment.\n\n3. **Absurd image**  \n   – The idea of a soda (a non‑living drink) going to work and applying for a job is silly and unexpected. That surprise adds to the humor.\n\n**So the joke is a pun:**\n\n- The soda is attracted to the office becau

# TIME TRAVEL

In [12]:
workflow.get_state({'configurable':{'thread_id':'1','checkpoint_id':'1f19f13a-5d6c-6fae-8000-3740798a5cd4'}})

StateSnapshot(values={'topic': 'cold drink'}, next=('joke_generator',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f19f13a-5d6c-6fae-8000-3740798a5cd4'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-23T16:56:56.070955+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f13a-5d6a-689f-bfff-8723deeaf980'}}, tasks=(PregelTask(id='8753569a-689d-d6b2-1448-ff4be617fa3c', name='joke_generator', path=('__pregel_pull', 'joke_generator'), error=None, interrupts=(), state=None, result={'joke': 'Why did the soda apply for a job?  \nBecause it heard the office had great *cool* vibes and wanted to *pop* into the team!'}),), interrupts=())

In [15]:
workflow.invoke(None,{'configurable':{'thread_id':'1','checkpoint_id':'1f19f13a-5d6c-6fae-8000-3740798a5cd4'}})

{'topic': 'cold drink',
 'joke': 'Why did the cold drink go to school?\n\nBecause it wanted to *cool* its grades!',
 'explaination': '**What the joke is doing**\n\n1. **Word “cool” has two meanings**  \n   * “Cool” as a temperature – something that is cold.  \n   * “Cool” as a slang word meaning “good, impressive, or better.”\n\n2. **The set‑up**  \n   * A “cold drink” (think of a chilled soda or iced coffee) is already literally cold.  \n   * The joke asks why it would go to school, which is a place where people try to get better “grades.”\n\n3. **The punch‑line**  \n   * It says the drink wants to *cool* its grades.  \n   * The humor comes from the double meaning:  \n     * The drink could “cool” itself down (it’s already cold).  \n     * It could also “cool” its grades, meaning make its grades better or “cool” (impressive).\n\nSo the joke is a play on the word “cool,” using it for both temperature and the idea of improving performance.'}

In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'cold drink', 'joke': 'Why did the cold drink go to school?\n\nBecause it wanted to *cool* its grades!', 'explaination': '**What the joke is doing**\n\n1. **Word “cool” has two meanings**  \n   * “Cool” as a temperature – something that is cold.  \n   * “Cool” as a slang word meaning “good, impressive, or better.”\n\n2. **The set‑up**  \n   * A “cold drink” (think of a chilled soda or iced coffee) is already literally cold.  \n   * The joke asks why it would go to school, which is a place where people try to get better “grades.”\n\n3. **The punch‑line**  \n   * It says the drink wants to *cool* its grades.  \n   * The humor comes from the double meaning:  \n     * The drink could “cool” itself down (it’s already cold).  \n     * It could also “cool” its grades, meaning make its grades better or “cool” (impressive).\n\nSo the joke is a play on the word “cool,” using it for both temperature and the idea of improving performance.'}, next=(), config={'config

# UPDATING STATE

In [19]:
workflow.update_state({'configurable':{'thread_id':'1','checkpoint_id':'1f19f13a-5d6c-6fae-8000-3740798a5cd4','checkpoint_ns':''}},{'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f19f195-a882-6d84-8001-6de4b38f2106'}}

In [ ]:
# Here we will pass the checkpoint id of 'samosa' acquired in the last step to generate a joke on it.

workflow.invoke(None,{'configurable':{'thread_id':1,'checkpoint_id':'1f19f195-a882-6d84-8001-6de4b38f2106'}})    

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy?\n\nBecause it kept feeling *crisp*ed out and wanted to *fill* in the gaps in its life!',
 'explaination': '**Short answer**\n\nThe joke is a pun that mixes two meanings of words that describe a samosa (the fried pastry).  \n- **“Crisped out”** sounds like “crisped out” (the samosa’s outer shell is crisp, and “crisped out” also means feeling tired or burnt out).  \n- **“Fill in the gaps”** plays on the fact that a samosa is a *filled* pastry, and “fill in the gaps” is what you do in therapy to address missing or broken parts of your life.\n\nSo the humor comes from pretending a food item has human feelings and wants to go to therapy just like a person would. The words that describe the samosa’s texture and filling are used as if they were emotional complaints.'}

In [ ]:
 # We accidently called the workflow for cold drink once again instead of samosa, so that's why we see two extra cold drink states too  

list(workflow.get_state_history(config = config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?\n\nBecause it kept feeling *crisp*ed out and wanted to *fill* in the gaps in its life!', 'explaination': '**Short answer**\n\nThe joke is a pun that mixes two meanings of words that describe a samosa (the fried pastry).  \n- **“Crisped out”** sounds like “crisped out” (the samosa’s outer shell is crisp, and “crisped out” also means feeling tired or burnt out).  \n- **“Fill in the gaps”** plays on the fact that a samosa is a *filled* pastry, and “fill in the gaps” is what you do in therapy to address missing or broken parts of your life.\n\nSo the humor comes from pretending a food item has human feelings and wants to go to therapy just like a person would. The words that describe the samosa’s texture and filling are used as if they were emotional complaints.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f19c-4d58-6262-8003-3501bc920cc2'}}, metadata={'